# Experiment 035b: 1.5B matched scale check (Colab)

Run of fixed seed 0 full and LoRA pair at A_GC. The full arm runs first, and evaluation waits until both training outcomes are settled

## 0. Compute units and Drive

In [ ]:
NOTEBOOK_NAME = '035_6_colab_b15_scale_check'
import glob, json, os, time
ALLOCATOR_CONFIG = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ALLOCATOR_CONFIG
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 370.41
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': NOTEBOOK_NAME,
                  'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)


session 20260819_203619 | cu_before = 370.41
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved open session: 20260819_203619


## 1. Runtime

In [ ]:
import shutil, subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 79, 'need the A100 80GB runtime'
assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == ALLOCATOR_CONFIG
print('local free:', round(shutil.disk_usage('/content').free / 1024**3, 1), 'GiB')
print('drive free:', round(shutil.disk_usage(ROOT).free / 1024**3, 1), 'GiB')


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-c5754400-9aa8-8cd2-1787-627726f0f419)
total vram: 79.3 GiB
local free: 188.6 GiB
drive free: 179.2 GiB


## 2. Repository and dependencies

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')


dependencies installed


In [ ]:
import getpass, os
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None
print('W&B key loaded')


W&B key loaded


## 3. Wiring checks

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_035_matched_grpo.py',
    'tests/test_training_wrapper_contracts.py',
    'tests/test_checkpoint.py',
    'tests/test_sentinel.py',
    'tests/test_protocol_freeze.py',
], check=True)
print('wiring checks passed')


wiring checks passed


In [ ]:
import json, os, shutil, sys, time
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s35_matched_grpo import (
    AGC_FULL_A1_ERROR, ALLOCATOR_CONFIG, ENDPOINT_MANIFEST, PILOT_RUNGS_035,
    TRAINING_COMPLETE, arm_tag, b15_arm_state, bind_start,
    check_pair_parity, commit_b15_a1_conclusion, commit_b15_drive_resume,
    commit_b15_pair_settled,
    commit_endpoint, commit_pair_parity, commit_start_probe, config_path,
    endpoint_state, prepare_b15_scale_check, safe_close_checkpoints,
)

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
RUNG = 'A_GC'
TECH = os.path.join(ROOT, 'results', '035_technical')
START_PATH = os.path.join(TECH, 'starts', 'B15', 'seed0.json')
full_cfg = load_config(config_path('B15', 0, 'full'))
lora_cfg = load_config(config_path('B15', 0, 'lora'))
assert full_cfg.log.wandb_project == lora_cfg.log.wandb_project == 'inm363-grpo'
os.environ['WANDB_PROJECT'] = full_cfg.log.wandb_project
start_record = bind_start(full_cfg, START_PATH)
authorisation = prepare_b15_scale_check(ROOT, START_PATH)
pilot = read_json(os.path.join(TECH, 'pilots', 'B15', RUNG, 'pilot_report.json'))
assert pilot.get('passed') is True
assert authorisation['execution'] == pilot['execution'] == PILOT_RUNGS_035[RUNG]
required_gib = 30 + 3 * pilot['checkpoint_bytes'] / 1024**3
assert shutil.disk_usage(ROOT).free / 1024**3 >= required_gib, (
    f'need at least {required_gib:.1f} GiB free on Drive')
print('pair authorised, Drive requirement:', round(required_gib, 1), 'GiB')


pair authorised, Drive requirement: 81.8 GiB


In [ ]:
def append_jsonl(path, row):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a') as f:
        f.write(json.dumps(row) + '\n')

def active_training_workers(cfg_path):
    process_list = subprocess.run(
        ['ps', '-eo', 'pid=,args='], capture_output=True, text=True, check=True).stdout
    return [line.strip() for line in process_list.splitlines()
            if 'scripts/s35_worker.py' in line and '--task train' in line
            and f'--config {cfg_path}' in line]

def worker_command(task, cfg_path, status_path, out_dir, result_dir, **kwargs):
    command = [
        sys.executable, 'scripts/s35_worker.py', '--task', task,
        '--config', cfg_path, '--start-record', START_PATH, '--rung', RUNG,
        '--status', status_path, '--out-dir', out_dir, '--result-dir', result_dir,
    ]
    for key, value in kwargs.items():
        if value is not None:
            command.extend(['--' + key.replace('_', '-'), str(value)])
    return command

def run_worker(command, status_path, result_dir, label, keep_status=False):
    if os.path.exists(status_path) and not keep_status:
        os.remove(status_path)
    started = time.time()
    process = stream_subprocess(command)
    status = read_json(status_path) if os.path.exists(status_path) else {
        'completed': False, 'error_type': 'MissingStatus',
        'error_message': 'worker exited without a status file'}
    status = {**status, 'label': label, 'returncode': process.returncode,
              'parent_wall_seconds': round(time.time() - started, 1)}
    append_jsonl(os.path.join(result_dir, '035b_attempts.jsonl'), status)
    append_jsonl(os.path.join(RESULTS, 'compute_ledger.jsonl'), {
        'run_tag': label, 'condition': f'035b_{status.get("task", "worker")}',
        'timings_sec': {'worker': status['parent_wall_seconds']},
        'train_sec': status['parent_wall_seconds'] if status.get('task') == 'train' else 0,
        'train_steps': status.get('completed_updates', 0),
        'completed': bool(status.get('completed')), 'gpu': status.get('gpu'),
    })
    return process, status


## 4. Pair parity

In [ ]:
def run_parity(mode):
    parity_dir = os.path.join(TECH, 'parity', 'B15', 'seed0', RUNG)
    record_path = os.path.join(parity_dir, f'{mode}_parity.json')
    if os.path.exists(record_path):
        return read_json(record_path)
    os.makedirs(parity_dir, exist_ok=True)
    local_out = f'/content/035b_parity_B15_{mode}_{RUNG}'
    if os.path.isdir(local_out):
        shutil.rmtree(local_out)
    status_path = os.path.join(parity_dir, f'{mode}_status.json')
    command = worker_command(
        'parity', config_path('B15', 0, mode), status_path,
        local_out, parity_dir, parity_record=record_path)
    try:
        process, status = run_worker(
            command, status_path, parity_dir, f'035b_parity_B15_{mode}_{RUNG}')
    finally:
        if os.path.isdir(local_out):
            shutil.rmtree(local_out)
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'pair parity failed for B15 {mode}: {status}')
    return read_json(record_path)

def run_pair_parity():
    path = os.path.join(TECH, 'parity', 'B15', 'seed0', RUNG, 'pair_parity.json')
    if os.path.exists(path):
        return check_pair_parity(path, 'B15')
    lora = run_parity('lora')
    full = run_parity('full')
    return commit_pair_parity(path, lora, full)

pair_parity = run_pair_parity()
print('pair parity:', pair_parity['checks'])


pair parity: {'lora_policy_vs_reference': 0.0, 'full_policy_vs_reference': 0.0, 'policy_between_modes': 0.0, 'reference_between_modes': 0.0}


## 5. Matched training

In [ ]:
def arm_paths(mode):
    tag = arm_tag('B15', 0, mode)
    return (os.path.join(ROOT, 'results', tag),
            os.path.join(ROOT, 'checkpoints', tag, 'grpo'))

def run_arm(mode):
    state = b15_arm_state(ROOT, mode)
    if state['state'] in ('completed', 'a1'):
        return state
    if state['state'] == 'a1_pending':
        record = commit_b15_a1_conclusion(ROOT, mode)
        return {'state': 'a1', 'record': record}
    if state['state'] == 'drive_interrupted_pending':
        commit_b15_drive_resume(ROOT)
        state = b15_arm_state(ROOT, mode)
    if state['state'] == 'failed':
        raise RuntimeError(f'the previous {mode} failure requires inspection')
    resume = bool(state.get('drive_resume'))

    cfg_path = config_path('B15', 0, mode)
    active = active_training_workers(cfg_path)
    assert not active, f'A_GC training worker already active: {active}'
    result_dir, output_dir = arm_paths(mode)
    os.makedirs(result_dir, exist_ok=True)
    status_path = os.path.join(result_dir, 'worker_status.json')
    precision_path = os.path.join(result_dir, 'precision_report.json')
    command = worker_command(
        'train', cfg_path, status_path, output_dir, result_dir,
        precision_report=precision_path)
    report_to = ['wandb'] if resume else ['tensorboard', 'wandb']
    command.extend(['--report-to', *report_to])
    process, status = run_worker(
        command, status_path, result_dir, f'035b_B15_seed0_{mode}_{RUNG}',
        keep_status=resume)
    if process.returncode == 0 and status.get('completed'):
        return b15_arm_state(ROOT, mode)
    if (status.get('cuda_oom') is False and status.get('error_type') == 'RuntimeError'
            and status.get('error_message') == AGC_FULL_A1_ERROR):
        record = commit_b15_a1_conclusion(ROOT, mode)
        return {'state': 'a1', 'record': record}
    raise RuntimeError(f'A_GC training failed for B15 seed 0 {mode}: {status}')

# settle the full outcome before the LoRA arm starts
full_outcome = run_arm('full')
lora_outcome = run_arm('lora')
settled = commit_b15_pair_settled(ROOT)
print('full:', full_outcome['state'])
print('LoRA:', lora_outcome['state'])


Streaming output truncated to the last 5000 lines.
│ │ an audience   │ marketing     │               │              │           │ │
│ │ to pass along │ entrusts the  │               │              │           │ │
│ │ the content   │ sharing of    │               │              │           │ │
│ │ to others:    │ content among │               │              │           │ │
│ │ A. Viral      │ users to      │               │              │           │ │
│ │ marketing.    │ peers.</thin… │               │              │           │ │
│ │ B.            │ <answer>D</a… │               │              │           │ │
│ │ Word-of-mouth │               │               │              │           │ │
│ │ marketing.    │               │               │              │           │ │
│ │ C.            │               │               │              │           │ │
│ │ Direct-respo… │               │               │              │           │ │
│ │ advertising.  │               │               │       

## 6. Evaluation

In [ ]:
def verify_start_probe(completed_modes):
    if not completed_modes:
        return None
    result_dir = os.path.join(ROOT, 'results', '035b_start_B15_seed0')
    manifest = os.path.join(result_dir, '035_start_probe_manifest.json')
    if os.path.exists(manifest):
        return commit_start_probe(result_dir, '/content/unused', start_record)
    mode = completed_modes[0]
    _, output_dir = arm_paths(mode)
    scratch = '/content/035b_start_probe_B15_seed0'
    if os.path.isdir(scratch):
        shutil.rmtree(scratch)
    os.makedirs(scratch)
    os.makedirs(result_dir, exist_ok=True)
    status_path = os.path.join(result_dir, 'eval_status.json')
    command = worker_command(
        'eval', config_path('B15', 0, mode), status_path,
        output_dir, result_dir, branch='start', scratch_dir=scratch)
    process, status = run_worker(
        command, status_path, result_dir, '035b_start_eval_B15_seed0')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'start probe failed: {status}')
    record = commit_start_probe(result_dir, scratch, start_record)
    shutil.rmtree(scratch)
    return record

def run_endpoint(mode, training):
    result_dir, output_dir = arm_paths(mode)
    state = endpoint_state(result_dir, training)
    if state == 'tampered':
        raise RuntimeError(f'committed B15 {mode} endpoint changed')
    if state == 'committed':
        return read_json(os.path.join(result_dir, ENDPOINT_MANIFEST))
    scratch = f'/content/035b_eval_B15_{mode}_{RUNG}'
    if os.path.isdir(scratch):
        shutil.rmtree(scratch)
    os.makedirs(scratch)
    status_path = os.path.join(result_dir, 'eval_status.json')
    command = worker_command(
        'eval', config_path('B15', 0, mode), status_path,
        output_dir, result_dir, branch=mode, scratch_dir=scratch)
    process, status = run_worker(
        command, status_path, result_dir, f'035b_eval_B15_seed0_{mode}_{RUNG}')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'endpoint evaluation failed for {mode}: {status}')
    record = commit_endpoint(
        result_dir, scratch, training, package='035b_scale_check')
    shutil.rmtree(scratch)
    safe_close_checkpoints(result_dir, output_dir)
    return record

states = {mode: b15_arm_state(ROOT, mode) for mode in ('full', 'lora')}
# evaluate only arms that completed all 500 updates
completed_modes = [mode for mode, row in states.items() if row['state'] == 'completed']
start_probe = verify_start_probe(completed_modes)
endpoints = {mode: run_endpoint(mode, states[mode]['record'])
             for mode in completed_modes}
print('start probe:', 'committed' if start_probe else 'not needed')
print('endpoints:', sorted(endpoints))


Loading weights: 100% 338/338 [00:00<00:00, 2736.68it/s]
start logprob (1884 items): 100% 118/118 [00:06<00:00, 17.51it/s, acc 0.695]
Loading weights: 100% 338/338 [00:02<00:00, 158.35it/s]
grpo eval (1884 items): 100% 118/118 [18:37<00:00,  9.47s/it, acc 0.604 fmt 0.929]
Loading weights: 100% 338/338 [00:02<00:00, 154.11it/s]
grpo logprob (1884 items): 100% 118/118 [00:06<00:00, 19.27it/s, acc 0.623]
Loading weights: 100% 338/338 [00:00<00:00, 2577.62it/s]
grpo eval (1884 items): 100% 118/118 [50:29<00:00, 25.68s/it, acc 0.661 fmt 0.952]
Loading weights: 100% 338/338 [00:00<00:00, 2583.86it/s]
grpo logprob (1884 items): 100% 118/118 [00:11<00:00, 10.26it/s, acc 0.692]
start probe: committed
endpoints: ['full', 'lora']


## 7. Current state

In [ ]:
states = {mode: b15_arm_state(ROOT, mode) for mode in ('full', 'lora')}
print('full outcome:', states['full']['state'])
print('LoRA outcome:', states['lora']['state'])
for mode, row in states.items():
    if row['state'] == 'completed':
        result_dir, _ = arm_paths(mode)
        print(mode, 'endpoint:', endpoint_state(result_dir, row['record']))
print('pair settled:', read_json(os.path.join(TECH, '035_b15_pair_settled.json'))['completed'])


## 8. Close compute units

In [ ]:
cu_after = None
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open session record')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)
